In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW:", DATA_RAW)

PROJECT_ROOT: c:\temp\python_learning\ml_projects\ml_projects_batch_01\05_bike_sharing_demand
DATA_RAW: c:\temp\python_learning\ml_projects\ml_projects_batch_01\05_bike_sharing_demand\data\raw


In [3]:
train = pd.read_csv(DATA_RAW / "train.csv")

print("train shape:", train.shape)
display(train.head())

train shape: (10886, 12)


,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


Ячейка 4 — parse datetime and create features

In [4]:
train["datetime"] = pd.to_datetime(train["datetime"])

train["year"] = train["datetime"].dt.year
train["month"] = train["datetime"].dt.month
train["day"] = train["datetime"].dt.day
train["hour"] = train["datetime"].dt.hour
train["weekday"] = train["datetime"].dt.weekday
train["is_weekend"] = train["weekday"].isin([5, 6]).astype(int)

display(train.head())

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,year,month,day,hour,weekday,is_weekend
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16,2011,1,1,0,5,1
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40,2011,1,1,1,5,1
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32,2011,1,1,2,5,1
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13,2011,1,1,3,5,1
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1,2011,1,1,4,5,1


In [5]:
target_col = "count"

leakage_cols = ["casual", "registered"]

excluded_cols = [
    target_col,
    "casual",
    "registered",
    "datetime",
    "day",  # excluded from first baseline because of transfer-risk
]

feature_cols = [col for col in train.columns if col not in excluded_cols]

print("Target:", target_col)
print("Excluded columns:", excluded_cols)
print("Feature columns:")
print(feature_cols)

Target: count
Excluded columns: ['count', 'casual', 'registered', 'datetime', 'day']
Feature columns:
['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'year', 'month', 'hour', 'weekday', 'is_weekend']


In [6]:
X = train[feature_cols].copy()
y = train[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
display(y.head())

X shape: (10886, 13)
y shape: (10886,)


,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,year,month,hour,weekday,is_weekend
0,1,0,0,1,9.84,14.395,81,0.0,2011,1,0,5,1
1,1,0,0,1,9.02,13.635,80,0.0,2011,1,1,5,1
2,1,0,0,1,9.02,13.635,80,0.0,2011,1,2,5,1
3,1,0,0,1,9.84,14.395,75,0.0,2011,1,3,5,1
4,1,0,0,1,9.84,14.395,75,0.0,2011,1,4,5,1


0    16
1    40
2    32
3    13
4     1
Name: count, dtype: int64

In [7]:
print("Leakage columns in X:")
print(set(X.columns).intersection(leakage_cols))

print("\nTarget in X:")
print(target_col in X.columns)

print("\nRaw datetime in X:")
print("datetime" in X.columns)

print("\nDay in first baseline X:")
print("day" in X.columns)

Leakage columns in X:
set()

Target in X:
False

Raw datetime in X:
False

Day in first baseline X:
False


Ячейка 8 — local validation split

In [8]:
local_train_mask = train["day"] <= 15
local_valid_mask = train["day"].between(16, 19)

X_train = X.loc[local_train_mask].copy()
y_train = y.loc[local_train_mask].copy()

X_valid = X.loc[local_valid_mask].copy()
y_valid = y.loc[local_valid_mask].copy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_valid shape:", y_valid.shape)

X_train shape: (8600, 13)
y_train shape: (8600,)
X_valid shape: (2286, 13)
y_valid shape: (2286,)


Ячейка 9 — validate split logic

In [9]:
print("Local train days:")
print(train.loc[local_train_mask, "day"].value_counts().sort_index())

print("\nLocal validation days:")
print(train.loc[local_valid_mask, "day"].value_counts().sort_index())

print("\nLocal train datetime range:")
print(train.loc[local_train_mask, "datetime"].min())
print(train.loc[local_train_mask, "datetime"].max())

print("\nLocal validation datetime range:")
print(train.loc[local_valid_mask, "datetime"].min())
print(train.loc[local_valid_mask, "datetime"].max())

Local train days:
day
1     575
2     573
3     573
4     574
5     575
6     572
7     574
8     574
9     575
10    572
11    568
12    573
13    574
14    574
15    574
Name: count, dtype: int64

Local validation days:
day
16    574
17    575
18    563
19    574
Name: count, dtype: int64

Local train datetime range:
2011-01-01 00:00:00
2012-12-15 23:00:00

Local validation datetime range:
2011-01-16 00:00:00
2012-12-19 23:00:00


## Stage 3 baseline setup notes

This notebook uses only `train.csv`.

Official Kaggle `test.csv` is not used for local validation because it has no `count`.

Target:

- `count`

Excluded from `X`:

- `count`
- `casual`
- `registered`
- raw `datetime`
- `day` for the first baseline

Leakage control:

- `casual` and `registered` are excluded because they are target components.
- `casual + registered == count`.
- raw `datetime` is parsed into derived features.
- first baseline excludes `day` because train contains days 1–19, while official test contains days 20–31.

Local validation split:

- local train: days 1–15
- local validation: days 16–19

No models are trained yet in this setup section.
No official test data is used.

## Baseline models

In [10]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error

Metrics helpers

In [14]:
def clipped_predictions(y_pred):
    return np.clip(y_pred, a_min=0, a_max=None)


def regression_metrics(y_true, y_pred):
    y_pred_clipped = clipped_predictions(y_pred)

    rmsle = np.sqrt(mean_squared_log_error(y_true, y_pred_clipped))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_clipped))
    mae = mean_absolute_error(y_true, y_pred_clipped)

    return {
        "RMSLE": rmsle,
        "RMSE": rmse,
        "MAE": mae,
        "min_pred_raw": np.min(y_pred),
        "min_pred_clipped": np.min(y_pred_clipped),
        "max_pred_raw": np.max(y_pred),
    }

Define models

In [13]:
models = {
    "DummyRegressor_mean": DummyRegressor(strategy="mean"),
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "DecisionTree_depth3": DecisionTreeRegressor(max_depth=3, random_state=42),
}

Fit on local train, evaluate on local validation

In [15]:
results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)

    metrics = regression_metrics(y_valid, y_pred)
    metrics["model"] = model_name
    results.append(metrics)

results_df = (
    pd.DataFrame(results)
    .set_index("model")
    .sort_values("RMSLE")
)

results_df

,RMSLE,RMSE,MAE,min_pred_raw,min_pred_clipped,max_pred_raw
model,,,,,,
DecisionTree_depth3,0.767086,130.765166,91.359668,15.143469,15.143469,375.426947
Ridge,1.242683,146.371947,106.992302,-123.156212,0.000000,438.767285
LinearRegression,1.242720,146.371005,106.992386,-123.176376,0.000000,438.779023
DummyRegressor_mean,1.531283,183.076977,142.644424,190.530233,190.530233,190.530233


### Baseline findings

- All models were trained only on local train rows: days 1–15.
- All models were evaluated only on local validation rows: days 16–19.
- Official Kaggle test.csv was not used.
- Predictions were clipped at 0 before RMSLE calculation.
- No hyperparameter tuning was performed.
- No `casual` or `registered` columns were used.

Best simple baseline by RMSLE:

- `<fill after results>`

Important note:

- This is an honest first baseline, not a final model.

## Optional experiment: baseline with day feature

`day` is available from `datetime`, so it is not target leakage.

However, it may transfer poorly because:

- train.csv contains days 1–19;
- official test.csv contains days 20–31.

This experiment is diagnostic only.

Feature set with day

In [16]:
feature_cols_with_day = feature_cols + ["day"]

print("Feature columns without day:")
print(feature_cols)

print("\nFeature columns with day:")
print(feature_cols_with_day)

print("\nLeakage columns in feature_cols_with_day:")
print(set(feature_cols_with_day).intersection(leakage_cols))

Feature columns without day:
['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'year', 'month', 'hour', 'weekday', 'is_weekend']

Feature columns with day:
['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'year', 'month', 'hour', 'weekday', 'is_weekend', 'day']

Leakage columns in feature_cols_with_day:
set()


Create X with day

In [17]:
X_with_day = train[feature_cols_with_day].copy()

X_train_day = X_with_day.loc[local_train_mask].copy()
X_valid_day = X_with_day.loc[local_valid_mask].copy()

print("X_train_day shape:", X_train_day.shape)
print("X_valid_day shape:", X_valid_day.shape)

X_train_day shape: (8600, 14)
X_valid_day shape: (2286, 14)


Evaluate same models with day

In [18]:
results_with_day = []

for model_name, model in models.items():
    model.fit(X_train_day, y_train)
    y_pred = model.predict(X_valid_day)

    metrics = regression_metrics(y_valid, y_pred)
    metrics["model"] = model_name
    metrics["feature_set"] = "with_day"
    results_with_day.append(metrics)

results_with_day_df = (
    pd.DataFrame(results_with_day)
    .set_index("model")
    .sort_values("RMSLE")
)

results_with_day_df

,RMSLE,RMSE,MAE,min_pred_raw,min_pred_clipped,max_pred_raw,feature_set
model,,,,,,,
DecisionTree_depth3,0.767086,130.765166,91.359668,15.143469,15.143469,375.426947,with_day
Ridge,1.243533,146.323182,107.644096,-119.697528,0.000000,441.394931,with_day
LinearRegression,1.243568,146.322253,107.644093,-119.717695,0.000000,441.406589,with_day
DummyRegressor_mean,1.531283,183.076977,142.644424,190.530233,190.530233,190.530233,with_day


Compare no-day vs with-day

In [19]:
results_no_day_df = results_df.copy()
results_no_day_df["feature_set"] = "without_day"

comparison_df = (
    pd.concat([
        results_no_day_df.reset_index(),
        results_with_day_df.reset_index()
    ])
    .set_index(["feature_set", "model"])
    .sort_values("RMSLE")
)

comparison_df

RMSLE        RMSE         MAE  \
feature_set model                                                   
without_day DecisionTree_depth3  0.767086  130.765166   91.359668   
with_day    DecisionTree_depth3  0.767086  130.765166   91.359668   
without_day Ridge                1.242683  146.371947  106.992302   
            LinearRegression     1.242720  146.371005  106.992386   
with_day    Ridge                1.243533  146.323182  107.644096   
            LinearRegression     1.243568  146.322253  107.644093   
without_day DummyRegressor_mean  1.531283  183.076977  142.644424   
with_day    DummyRegressor_mean  1.531283  183.076977  142.644424   

                                 min_pred_raw  min_pred_clipped  max_pred_raw  
feature_set model                                                              
without_day DecisionTree_depth3     15.143469         15.143469    375.426947  
with_day    DecisionTree_depth3     15.143469         15.143469    375.426947  
without_day Ridge                 -123.156212          0.000000    438.767285  
            LinearRegression      -123.176376          0.000000    438.779023  
with_day    Ridge                 -119.697528          0.000000    441.394931  
            LinearRegression      -119.717695          0.000000    441.406589  
without_day DummyRegressor_mean    190.530233        190.530233    190.530233  
with_day    DummyRegressor_mean    190.530233        190.530233    190.530233

### Optional `day` experiment findings

- `day` was tested only as an optional diagnostic feature.
- `day` is not target leakage because it is derived from known `datetime`.
- However, it may not transfer well to official test because local train/validation uses days 1–19, while official test uses days 20–31.
- The main baseline conclusion should prioritize the safer no-day feature set unless there is a strong reason to include `day`.

## Stage 3 baseline conclusions

### Task

Time-aware regression.

Target:

- `count`

Primary metric:

- RMSLE

Additional metrics:

- RMSE
- MAE

### Data used

Only `train.csv` was used for local training and validation.

Official Kaggle `test.csv` was not used for validation because it does not contain `count`.

### Feature set

Main baseline feature set:

- `season`
- `holiday`
- `workingday`
- `weather`
- `temp`
- `atemp`
- `humidity`
- `windspeed`
- `year`
- `month`
- `hour`
- `weekday`
- `is_weekend`

Excluded from `X`:

- `count`
- `casual`
- `registered`
- raw `datetime`
- `day` in the main baseline

`casual` and `registered` are excluded because they are target components:

```text
casual + registered == count
Local validation split

Time-aware local validation:

local train: days 1–15
local validation: days 16–19

This split is more appropriate than random split because official Kaggle test contains later days of each month.

Models compared

No hyperparameter tuning was performed.

Models:

DummyRegressor(strategy="mean")
LinearRegression
Ridge
DecisionTreeRegressor(max_depth=3, random_state=42)
Main baseline metrics

Main feature set without day:

model	RMSLE	RMSE	MAE
DecisionTree_depth3	0.767086	130.765166	91.359668
Ridge	1.242683	146.371947	106.992302
LinearRegression	1.242720	146.371005	106.992386
DummyRegressor_mean	1.531283	183.076977	142.644424

Best simple baseline by RMSLE:

DecisionTreeRegressor(max_depth=3)
Optional day experiment

day was tested as an optional diagnostic feature.

Result:

day did not improve the best RMSLE.
Best model remained DecisionTreeRegressor(max_depth=3).
Linear models slightly worsened by RMSLE with day.

Therefore, the main baseline remains the safer no-day feature set.

Prediction clipping

Predictions were clipped at 0 before RMSLE calculation.

Reason:

RMSLE requires non-negative predictions.
Negative rental demand is impossible.

Raw negative predictions were observed for:

LinearRegression
Ridge

This is an important model diagnostic.

Leakage checks

Passed:

casual not in X
registered not in X
count not in X
raw datetime not in X
official Kaggle test.csv not used for validation
no random split used as the main validation scheme
no hyperparameter tuning performed